# 02 — Simple RAG

## Objective

This notebook implements a basic Retrieval-Augmented Generation (RAG) pipeline using the corpus and FAISS index created in `01_build_corpus_and_index.ipynb`.

The notebook loads the existing data and index, retrieves relevant text chunks for a question, and uses the Qwen2.5-7B-Instruct language model to generate an answer based on the retrieved evidence.

## Pipeline

1. Load the existing corpus, FAISS index, and evaluation questions.
2. Initialize the retriever using the existing embedding model and index.
3. Retrieve the top-k relevant text chunks for each question.
4. Build a prompt using the retrieved evidence.
5. Generate answers with the language model.
6. Evaluate the generated answers and save the results.

## Experimental Setup

- **Dataset:** HotpotQA
- **Embedding model:** `sentence-transformers/all-MiniLM-L6-v2`
- **Vector index:** FAISS `IndexFlatIP`
- **Language model:** Qwen2.5-7B-Instruct (8-bit quantization)
- **Number of evaluation questions:** 10
- **Top-k retrieval:** 5

**Note:** This notebook reuses the corpus and FAISS index created in Notebook 1. It does not rebuild the corpus or regenerate document embeddings.

## Project Configuration


In [ ]:
# Install the required packages
%pip install -q -U faiss-cpu sentence-transformers bitsandbytes accelerate transformers

In [ ]:
# ============================================================
# Cell 1 — Project Configuration
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Basic_RAG")

DATASET_NAME = "hotpotqa"
TOP_K = 5
N_EVAL_SAMPLES = 10

DATA_DIR = PROJECT_ROOT / "data" / DATASET_NAME
CORPUS_DIR = PROJECT_ROOT / "corpus" / DATASET_NAME
INDEX_DIR = PROJECT_ROOT / "indexes" / DATASET_NAME
RESULTS_DIR = PROJECT_ROOT / "results" / DATASET_NAME

EVALUATION_FILE = DATA_DIR / "evaluation_10.json"
CORPUS_FILE = CORPUS_DIR / "corpus.json"
FAISS_INDEX_FILE = INDEX_DIR / "faiss.index"

print("Project configuration loaded.")
print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset: {DATASET_NAME}")
print(f"Top-k: {TOP_K}")
print(f"Evaluation samples: {N_EVAL_SAMPLES}")

Project configuration loaded.
Project root: /content/drive/MyDrive/Basic_RAG
Dataset: hotpotqa
Top-k: 5
Evaluation samples: 10


## Connect Google Drive and Check Required Files


In [ ]:
# ============================================================
# Cell 2 — Connect Google Drive and Check Required Files
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

required_files = [
    EVALUATION_FILE,
    CORPUS_FILE,
    FAISS_INDEX_FILE,
]

missing_files = []

print("Checking required files...")

for file_path in required_files:
    if file_path.exists():
        print(f"[OK] {file_path}")
    else:
        print(f"[MISSING] {file_path}")
        missing_files.append(str(file_path))

if missing_files:
    raise FileNotFoundError(
        "Required files are missing:\n"
        + "\n".join(missing_files)
    )

print("All required files are available.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checking required files...
[OK] /content/drive/MyDrive/Basic_RAG/data/hotpotqa/evaluation_10.json
[OK] /content/drive/MyDrive/Basic_RAG/corpus/hotpotqa/corpus.json
[OK] /content/drive/MyDrive/Basic_RAG/indexes/hotpotqa/faiss.index
All required files are available.


## Load the Existing Corpus and FAISS Index


In [ ]:
# ============================================================
# Cell 3 — Load Existing Corpus and FAISS Index
# ============================================================

import json
import faiss
import numpy as np

# Load corpus
with open(CORPUS_FILE, "r", encoding="utf-8") as file:
    corpus = json.load(file)

# Load FAISS index
faiss_index = faiss.read_index(str(FAISS_INDEX_FILE))

# Load evaluation questions
with open(EVALUATION_FILE, "r", encoding="utf-8") as file:
    evaluation_data = json.load(file)

print("Existing data loaded successfully.")
print(f"Number of corpus chunks: {len(corpus)}")
print(f"Number of vectors in FAISS index: {faiss_index.ntotal}")
print(f"Embedding dimension: {faiss_index.d}")
print(f"Number of evaluation samples: {len(evaluation_data)}")

Existing data loaded successfully.
Number of corpus chunks: 995
Number of vectors in FAISS index: 995
Embedding dimension: 384
Number of evaluation samples: 10


## Prepare the Retriever


The `FAISSRetriever` class below is copied from Notebook 1 (`01_build_corpus_and_index.ipynb`). It is reused here to retrieve relevant text chunks from the existing FAISS index without rebuilding the corpus or generating new document embeddings.

This duplication is intentional, to keep this notebook self-contained and easy to run independently. The class can be moved to a shared Python module in a future refactoring step.

In [ ]:
# ============================================================
# Cell 4 — Load Embedding Model and Initialize Retriever
# ============================================================

from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

class FAISSRetriever:
    """
    Simple reusable retriever based on:
    SentenceTransformer + FAISS IndexFlatIP
    """

    def __init__(self, embedding_model, index, corpus):
        self.embedding_model = embedding_model
        self.index = index
        self.corpus = corpus

    def retrieve(self, query, top_k=5):
        """
        Retrieve the top-k most similar chunks for a query.
        """

        # Encode query
        query_embedding = self.embedding_model.encode(
            [query],
            normalize_embeddings=True
        )

        query_embedding = np.asarray(
            query_embedding,
            dtype="float32"
        )

        # Search FAISS
        scores, indices = self.index.search(
            query_embedding,
            top_k
        )

        # Build readable results
        results = []

        for rank, (score, idx) in enumerate(
            zip(scores[0], indices[0]),
            start=1
        ):
            chunk = self.corpus[idx]

            result = {
                "rank": rank,
                "score": float(score),
                "chunk_id": chunk["chunk_id"],
                "sample_id": chunk["sample_id"],
                "title": chunk["title"],
                "text": chunk["text"],
                "dataset": chunk["dataset"],
                "source_type": chunk["source_type"]
            }

            results.append(result)

        return results

retriever = FAISSRetriever(
    embedding_model=embedding_model,
    index=faiss_index,
    corpus=corpus
)

print("Retriever initialized successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Retriever initialized successfully.


## Test Retrieval Before Using the LLM

Run retrieval for one question and inspect the returned passages, IDs, titles, and scores.

This is a retrieval-only smoke test; it should not call the language model.

In [ ]:
# ============================================================
# Cell 5 — Test Retrieval Before Using LLM
# ============================================================

test_sample = evaluation_data[0]
query = test_sample["question"]

retrieved_chunks = retriever.retrieve(
    query=query,
    top_k=TOP_K
)

print(f"Question: {query}")
print(f"\nNumber of retrieved chunks: {len(retrieved_chunks)}")

for result in retrieved_chunks:
    print("\n" + "-" * 60)
    print(f"Rank: {result['rank']}")
    print(f"Score: {result['score']:.4f}")
    print(f"Chunk ID: {result['chunk_id']}")
    print(f"Title: {result['title']}")
    print(f"Text: {result['text'][:500]}")

Question: What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?

Number of retrieved chunks: 5

------------------------------------------------------------
Rank: 1
Score: 0.5720
Chunk ID: 5a7a567255429941d65f25bd_chunk_9
Title: Iqbal F. Qadir
Text: Vice-Admiral Iqbal Fazl Quadir (Urdu:اقبال فضل قادر) , is a retired three-star rank admiral in the Pakistan Navy, former diplomat, and a defence analyst.  He is renown for his participation in second war with India when he was part of the flotilla that attacked the radar station in Dwarka, India.

------------------------------------------------------------
Rank: 2
Score: 0.5118
Chunk ID: 5a7a567255429941d65f25bd_chunk_4
Title: Mukachevo Radar Station
Text: Mukachevo radar station was a Soviet radar station providing early warning of ballistic missile attack.  It was located in Shipka in the far south west of Ukraine and was part of the Soviet, and then Russian 

## Build the RAG Prompt

Combine the user's question with the retrieved passages. The prompt should instruct the model to answer using the supplied evidence and to say when the evidence is insufficient.


In [ ]:
# ============================================================
# Cell 6 — Build RAG Prompt
# ============================================================

def build_rag_prompt(query, retrieved_chunks):
    context_parts = []

    for result in retrieved_chunks:
        context_parts.append(
            f"[Source: {result['title']} | "
            f"Chunk ID: {result['chunk_id']}]\n"
            f"{result['text']}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""You are a helpful question-answering assistant.
Answer the question using only the provided context.
If the context does not contain enough information to answer,
clearly state that the available information is insufficient.
Do not invent facts.

Context:
{context}

Question:
{query}

Answer:"""

    return prompt


rag_prompt = build_rag_prompt(
    query=query,
    retrieved_chunks=retrieved_chunks
)

print("RAG prompt created successfully.")
print(f"Prompt length: {len(rag_prompt)} characters")
print("\nPrompt preview:\n")
print(rag_prompt[:2000])

RAG prompt created successfully.
Prompt length: 3088 characters

Prompt preview:

You are a helpful question-answering assistant.
Answer the question using only the provided context.
If the context does not contain enough information to answer,
clearly state that the available information is insufficient.
Do not invent facts.

Context:
[Source: Iqbal F. Qadir | Chunk ID: 5a7a567255429941d65f25bd_chunk_9]
Vice-Admiral Iqbal Fazl Quadir (Urdu:اقبال فضل قادر) , is a retired three-star rank admiral in the Pakistan Navy, former diplomat, and a defence analyst.  He is renown for his participation in second war with India when he was part of the flotilla that attacked the radar station in Dwarka, India.

[Source: Mukachevo Radar Station | Chunk ID: 5a7a567255429941d65f25bd_chunk_4]
Mukachevo radar station was a Soviet radar station providing early warning of ballistic missile attack.  It was located in Shipka in the far south west of Ukraine and was part of the Soviet, and then Russian missil

## Connect the Existing LLM Interface

Import `call_llm(prompt)` from `qwen_model.py`.


In [ ]:
# ============================================================
# Cell 7 — Connect Existing LLM Interface
# ============================================================

import sys

MODEL_MODULE_DIR = Path("/content/drive/MyDrive")

if str(MODEL_MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_MODULE_DIR))

from qwen_model import call_llm

print("LLM interface imported successfully.")

Loading Qwen tokenizer...
Model path: /content/drive/MyDrive/Model_qwen2.5-7B-instruct/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading Qwen2.5-7B-Instruct in 8-bit...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Qwen model loaded successfully.
Device: cuda:0
VRAM allocated: 8.81 GB
LLM interface imported successfully.


## Run One End-to-End Simple RAG Query

We will start with a single question.

In [ ]:
# ============================================================
# Cell 8 — Run One End-to-End Simple RAG Query
# ============================================================

answer = call_llm(rag_prompt)

print("Question:")
print(query)

print("\nGenerated Answer:")
print(answer)

print("\nRetrieved Sources:")

for result in retrieved_chunks:
    print(
        f"- {result['title']} "
        f"(Chunk ID: {result['chunk_id']}, "
        f"Score: {result['score']:.4f})"
    )

Question:
What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?

Generated Answer:
The available information does not specify the location of the radar station attacked by Vice-Admiral Iqbal Fazl Quadir during the second war with India. The context mentions his participation in an attack on a radar station in Dwarka, India, but it does not provide details about its exact location on the Okhamandal Peninsula or confirm the type of radar station. Therefore, based on the given information, it is not possible to determine what Iqbal F. Qadir was on during this attack.

Retrieved Sources:
- Iqbal F. Qadir (Chunk ID: 5a7a567255429941d65f25bd_chunk_9, Score: 0.5720)
- Mukachevo Radar Station (Chunk ID: 5a7a567255429941d65f25bd_chunk_4, Score: 0.5118)
- Sevastopol Radar Station (Chunk ID: 5a7a567255429941d65f25bd_chunk_8, Score: 0.4814)
- Radar Station B-71 (Chunk ID: 5a7a567255429941d65f25bd_chunk_6, Score: 0.4713

## Run a Small Experiment and Save Results

Run Simple RAG Evaluation on 10 samples.

In [ ]:
# ============================================================
# Cell 9 — Run Simple RAG Evaluation
# ============================================================

import time

evaluation_results = []

for i, sample in enumerate(evaluation_data[:N_EVAL_SAMPLES], start=1):
    question = sample["question"]
    reference_answer = sample["answer"]

    # Retrieve relevant context
    retrieved_chunks = retriever.retrieve(
        query=question,
        top_k=TOP_K
    )

    # Build the RAG prompt
    prompt = build_rag_prompt(
        query=question,
        retrieved_chunks=retrieved_chunks
    )

    # Generate the answer and measure execution time
    start_time = time.time()
    generated_answer = call_llm(prompt)
    elapsed_time = time.time() - start_time

    # Store the result
    result = {
        "question_id": sample.get("id", i),
        "question": question,
        "reference_answer": reference_answer,
        "generated_answer": generated_answer,
        "retrieved_chunks": retrieved_chunks,
        "generation_time_seconds": round(elapsed_time, 2)
    }

    evaluation_results.append(result)

    print(f"Completed {i}/{min(N_EVAL_SAMPLES, len(evaluation_data))}")
    print(f"Generation time: {elapsed_time:.2f} seconds")
    print("-" * 60)

print("\nEvaluation completed.")
print(f"Total evaluated questions: {len(evaluation_results)}")


Completed 1/10
Generation time: 15.43 seconds
------------------------------------------------------------
Completed 2/10
Generation time: 6.29 seconds
------------------------------------------------------------
Completed 3/10
Generation time: 8.77 seconds
------------------------------------------------------------
Completed 4/10
Generation time: 4.83 seconds
------------------------------------------------------------
Completed 5/10
Generation time: 11.34 seconds
------------------------------------------------------------
Completed 6/10
Generation time: 10.32 seconds
------------------------------------------------------------
Completed 7/10
Generation time: 6.07 seconds
------------------------------------------------------------
Completed 8/10
Generation time: 6.26 seconds
------------------------------------------------------------
Completed 9/10
Generation time: 3.09 seconds
------------------------------------------------------------
Completed 10/10
Generation time: 6.85 secon

## Save Evaluation Results

In [ ]:
# ============================================================
# Cell 10 — Save Evaluation Results
# ============================================================

RESULTS_DIR = (
    Path(PROJECT_ROOT)
    / "results"
    / DATASET_NAME
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_FILE = RESULTS_DIR / "simple_rag_results.json"

with open(RESULTS_FILE, "w", encoding="utf-8") as file:
    json.dump(
        evaluation_results,
        file,
        ensure_ascii=False,
        indent=2
    )

print("Evaluation results saved successfully.")
print(f"File path: {RESULTS_FILE}")
print(f"Number of saved results: {len(evaluation_results)}")


Evaluation results saved successfully.
File path: /content/drive/MyDrive/Basic_RAG/results/hotpotqa/simple_rag_results.json
Number of saved results: 10


## Review the Results


In [ ]:
# ============================================================
# Cell 11 — Inspect Generated and Reference Answers
# ============================================================

for i, result in enumerate(evaluation_results, start=1):
    print(f"Question {i}")
    print("-" * 80)

    print("Question:")
    print(result["question"])

    print("\nReference Answer:")
    print(result["reference_answer"])

    print("\nGenerated Answer:")
    print(result["generated_answer"])

    print("\nGeneration Time:")
    print(f"{result['generation_time_seconds']:.2f} seconds")

    print("\n" + "=" * 80 + "\n")


Question 1
--------------------------------------------------------------------------------
Question:
What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?

Reference Answer:
flotilla

Generated Answer:
The available information does not specify the location of the radar station attacked by Vice-Admiral Iqbal Fazl Quadir during the second war with India. The context mentions his participation in an attack on a radar station in Dwarka, India, but it does not provide the exact location on the western shore of the Okhamandal Peninsula. Therefore, the available information is insufficient to answer the question.

Generation Time:
15.43 seconds


Question 2
--------------------------------------------------------------------------------
Question:
When did the park at which Tivolis Koncertsal is located open?

Reference Answer:
15 August 1843

Generated Answer:
Tivoli Gardens, the park at which Tivolis Koncertsal

In [ ]:
# ============================================================
# Cell 12 — Calculate Exact Match and Token-level F1
# ============================================================

import re
import string

def normalize_answer(text):
    """Normalize an answer using common HotpotQA conventions."""
    text = str(text).lower()
    text = "".join(
        char for char in text
        if char not in string.punctuation
    )
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = " ".join(text.split())
    return text


def calculate_exact_match(prediction, reference):
    return int(
        normalize_answer(prediction)
        == normalize_answer(reference)
    )


def calculate_token_f1(prediction, reference):
    prediction_tokens = normalize_answer(prediction).split()
    reference_tokens = normalize_answer(reference).split()

    if not prediction_tokens or not reference_tokens:
        return float(prediction_tokens == reference_tokens)

    common_tokens = set(prediction_tokens) & set(reference_tokens)

    # Count repeated tokens correctly
    from collections import Counter
    common_count = sum(
        (Counter(prediction_tokens) & Counter(reference_tokens)).values()
    )

    if common_count == 0:
        return 0.0

    precision = common_count / len(prediction_tokens)
    recall = common_count / len(reference_tokens)

    return 2 * precision * recall / (precision + recall)


# Calculate metrics for each question
for result in evaluation_results:
    result["exact_match"] = calculate_exact_match(
        result["generated_answer"],
        result["reference_answer"]
    )
    result["token_f1"] = calculate_token_f1(
        result["generated_answer"],
        result["reference_answer"]
    )

# Aggregate results
num_questions = len(evaluation_results)

mean_em = (
    sum(result["exact_match"] for result in evaluation_results)
    / num_questions
)

mean_f1 = (
    sum(result["token_f1"] for result in evaluation_results)
    / num_questions
)

print(f"Number of evaluated questions: {num_questions}")
print(f"Exact Match: {mean_em:.4f} ({mean_em * 100:.2f}%)")
print(f"Mean Token-level F1: {mean_f1:.4f} ({mean_f1 * 100:.2f}%)")

print("\nPer-question metrics:")
for i, result in enumerate(evaluation_results, start=1):
    print(
        f"Q{i}: EM={result['exact_match']}, "
        f"F1={result['token_f1']:.4f}"
    )


Number of evaluated questions: 10
Exact Match: 0.0000 (0.00%)
Mean Token-level F1: 0.1231 (12.31%)

Per-question metrics:
Q1: EM=0, F1=0.0000
Q2: EM=0, F1=0.3529
Q3: EM=0, F1=0.0645
Q4: EM=0, F1=0.0000
Q5: EM=0, F1=0.0833
Q6: EM=0, F1=0.0000
Q7: EM=0, F1=0.2222
Q8: EM=0, F1=0.2000
Q9: EM=0, F1=0.3077
Q10: EM=0, F1=0.0000


In [ ]:
# ============================================================
# Cell 13 — Check Reference Answer in Retrieved Context
# ============================================================

for i, result in enumerate(evaluation_results, start=1):
    reference = normalize_answer(result["reference_answer"])

    retrieved_context = " ".join(
        chunk["text"] for chunk in result["retrieved_chunks"]
    )
    normalized_context = normalize_answer(retrieved_context)

    answer_found = bool(reference) and reference in normalized_context

    result["reference_answer_in_context"] = answer_found

    print(f"Question {i}")
    print(f"Reference answer: {result['reference_answer']}")
    print(f"Answer found in retrieved context: {answer_found}")
    print("-" * 60)

found_count = sum(
    result["reference_answer_in_context"]
    for result in evaluation_results
)

print("\nRetrieval Context Coverage")
print(f"Reference answers found: {found_count}/{len(evaluation_results)}")
print(
    f"Coverage: "
    f"{found_count / len(evaluation_results) * 100:.2f}%"
)


Question 1
Reference answer: flotilla
Answer found in retrieved context: True
------------------------------------------------------------
Question 2
Reference answer: 15 August 1843
Answer found in retrieved context: True
------------------------------------------------------------
Question 3
Reference answer: Armenian
Answer found in retrieved context: True
------------------------------------------------------------
Question 4
Reference answer: Centre of Excellence
Answer found in retrieved context: True
------------------------------------------------------------
Question 5
Reference answer: Saoirse Ronan
Answer found in retrieved context: True
------------------------------------------------------------
Question 6
Reference answer: The Lorax
Answer found in retrieved context: True
------------------------------------------------------------
Question 7
Reference answer: Scott Adkins
Answer found in retrieved context: True
------------------------------------------------------------

In [ ]:
# ============================================================
# Cell 14 — Locate Reference Answers in Retrieved Chunks
# ============================================================

for i, result in enumerate(evaluation_results, start=1):
    reference = normalize_answer(result["reference_answer"])

    print(f"\nQuestion {i}: {result['question']}")
    print(f"Reference answer: {result['reference_answer']}")

    matches = []

    for chunk in result["retrieved_chunks"]:
        chunk_text = normalize_answer(chunk["text"])

        if reference and reference in chunk_text:
            matches.append(chunk)

    if matches:
        for chunk in matches:
            print(
                f"  Match at rank {chunk['rank']} | "
                f"Score: {chunk['score']:.4f} | "
                f"Title: {chunk['title']} | "
                f"Chunk ID: {chunk['chunk_id']}"
            )
    else:
        print("  No exact normalized answer match in retrieved chunks.")



Question 1: What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?
Reference answer: flotilla
  Match at rank 1 | Score: 0.5720 | Title: Iqbal F. Qadir | Chunk ID: 5a7a567255429941d65f25bd_chunk_9

Question 2: When did the park at which Tivolis Koncertsal is located open?
Reference answer: 15 August 1843
  Match at rank 2 | Score: 0.6321 | Title: Tivoli Gardens | Chunk ID: 5abca1a55542993a06baf937_chunk_1

Question 3: What is the shared country of ancestry between Art Laboe and Scout Tufankjian?
Reference answer: Armenian
  Match at rank 1 | Score: 0.5633 | Title: Scout Tufankjian | Chunk ID: 5a73977d554299623ed4ac08_chunk_8
  Match at rank 2 | Score: 0.3682 | Title: Art Laboe | Chunk ID: 5a73977d554299623ed4ac08_chunk_0

Question 4: The school in which the Wilmslow Show is held is designated as what?
Reference answer: Centre of Excellence
  Match at rank 2 | Score: 0.6463 | Title: Wilmslow High School | Ch

In [ ]:
# ============================================================
# Cell 15 — Summarize Simple RAG Evaluation
# ============================================================

num_questions = len(evaluation_results)

total_generation_time = sum(
    result["generation_time_seconds"]
    for result in evaluation_results
)

mean_generation_time = (
    total_generation_time / num_questions
    if num_questions else 0
)

answer_coverage = sum(
    result["reference_answer_in_context"]
    for result in evaluation_results
)

exact_match_count = sum(
    result["exact_match"]
    for result in evaluation_results
)

mean_f1 = (
    sum(result["token_f1"] for result in evaluation_results)
    / num_questions
    if num_questions else 0
)

print("Simple RAG Evaluation Summary")
print("=" * 50)
print(f"Number of questions: {num_questions}")
print(f"Reference answers found in retrieved context: "
      f"{answer_coverage}/{num_questions}")
print(f"Reference answer coverage: "
      f"{answer_coverage / num_questions * 100:.2f}%")
print(f"Exact Match: {exact_match_count / num_questions * 100:.2f}%")
print(f"Mean Token-level F1: {mean_f1 * 100:.2f}%")
print(f"Total generation time: {total_generation_time:.2f} seconds")
print(f"Average generation time: {mean_generation_time:.2f} seconds")


Simple RAG Evaluation Summary
Number of questions: 10
Reference answers found in retrieved context: 9/10
Reference answer coverage: 90.00%
Exact Match: 0.00%
Mean Token-level F1: 12.31%
Total generation time: 79.25 seconds
Average generation time: 7.92 seconds


## Results and Limitations

### Results

The Simple RAG pipeline was evaluated on 10 HotpotQA questions. The retriever returned context containing the reference answer for 9 out of 10 questions (90% reference answer coverage).

The baseline achieved an Exact Match score of 0% and a mean token-level F1 score of 12.31%. Total generation time was 79.25 seconds, with an average of 7.92 seconds per question.

These results indicate that retrieving relevant evidence does not necessarily guarantee that the language model will generate an answer matching the reference answer.

### Limitations

- **Small evaluation set:** The evaluation uses only 10 questions, so the results are preliminary and may not represent performance on the full HotpotQA dataset.
- **Answer generation:** The retrieved context contained the reference answer for 9 questions, but the baseline achieved low answer-matching scores. This suggests that answer generation and answer formatting should be investigated alongside retrieval quality.
- **Limited evaluation metrics:** Exact Match and token-level F1 measure textual overlap and may not fully capture semantically correct answers.
- **Retrieval coverage:** The reference answer was not found in the retrieved context for one question, indicating that retrieval can also be a source of error.

### Next Step

The next notebook explores alternative prompt designs and retrieval settings to investigate how they affect answer quality.